# SMAP SSS Icechunk Append Demo

Reads the SMAP_RSS_L3_SSS_SMI_8DAY-RUNNINGMEAN_V6 virtual Icechunk store and visualises its current state.
Run the top cells to snapshot **before**, send SQS messages in the background, then re-run the **After** cells to see the store grow.

In [ ]:
import os
import warnings
import earthaccess
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore")

# ---- CONFIG ----
STORE_BUCKET  = "podaac-sit-services-cloud-optimizer"
STORE_PREFIX  = "virtual_collections/_notebook_demo/SMAP_RSS_L3_SSS_SMI_8DAY-RUNNINGMEAN_V6.icechunk_v2.https/"
VCC_PREFIX    = "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/"
AWS_PROFILE   = "cumulus-sit"
REGION        = "us-west-2"

os.environ["AWS_PROFILE"] = AWS_PROFILE


def open_store():
    storage = icechunk.s3_storage(bucket=STORE_BUCKET, prefix=STORE_PREFIX, region=REGION)
    edl_token = earthaccess.get_edl_token()
    config = icechunk.Repository.fetch_config(storage)
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            VCC_PREFIX,
            icechunk.http_store(
                headers={"Authorization": f"Bearer {edl_token['access_token']}"}
            ),
        )
    )
    repo = icechunk.Repository.open(
        storage, config=config,
        authorize_virtual_chunk_access={VCC_PREFIX: icechunk.credentials.HttpAccess},
    )
    return xr.open_zarr(repo.readonly_session(branch="main").store,
                        consolidated=False, decode_times=True)


def plot_sss(ds, title):
    sss = ds["sss_smap"].isel(time=-1).values
    t   = str(ds.time.values[-1])[:10]
    n   = ds.sizes["time"]
    fig, ax = plt.subplots(figsize=(14, 6))
    img = ax.imshow(sss, origin="lower", aspect="auto",
                    extent=[-180, 180, -90, 90],
                    cmap="RdYlBu_r", vmin=30, vmax=38, interpolation="nearest")
    plt.colorbar(img, ax=ax, label="SSS (psu)", shrink=0.7)
    ax.set_title(f"{title}  |  last step: {t}  ({n} total steps)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()
    plt.show()


def plot_time_axis(ds, new_from=None, title="Time axis"):
    times = ds["time"].values
    fig, ax = plt.subplots(figsize=(12, 2))
    if new_from is not None:
        ax.scatter(times[:new_from], np.ones(new_from), s=10,
                   color="steelblue", alpha=0.5, label=f"Original ({new_from})", zorder=2)
        ax.scatter(times[new_from:], np.ones(len(times) - new_from), s=40,
                   color="tomato", alpha=1.0, label=f"Appended ({len(times) - new_from})", zorder=3)
        ax.legend(loc="upper left")
    else:
        ax.scatter(times, np.ones(len(times)), s=10, color="steelblue", alpha=0.6)
    ax.set_yticks([])
    ax.set_xlabel("Time")
    ax.set_title(f"{title}  ({len(times)} steps)")
    plt.tight_layout()
    plt.show()

In [ ]:
auth = earthaccess.login()
print("Authenticated:", auth.authenticated)

## Before — current store state

In [ ]:
ds_before = open_store()
n_before  = ds_before.sizes["time"]
print(f"{n_before} steps  |  {ds_before.time.values[0]} → {ds_before.time.values[-1]}")
plot_sss(ds_before, "SMAP SSS — BEFORE")
plot_time_axis(ds_before, title="Time axis BEFORE")

---
▶ **Send your SQS messages in the background now, then run the cells below.**

---

## After — re-read store once append has landed

In [ ]:
ds_after = open_store()
n_after  = ds_after.sizes["time"]
print(f"{n_after} steps  (+{n_after - n_before})  |  {ds_after.time.values[0]} → {ds_after.time.values[-1]}")
plot_sss(ds_after, "SMAP SSS — AFTER")
plot_time_axis(ds_after, new_from=n_before, title="Time axis AFTER")

### Side-by-side comparison

In [ ]:
sss_before = ds_before["sss_smap"].isel(time=-1).values
sss_after  = ds_after["sss_smap"].isel(time=-1).values
t_before   = str(ds_before.time.values[-1])[:10]
t_after    = str(ds_after.time.values[-1])[:10]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, data, label, t, n in [
    (axes[0], sss_before, "BEFORE", t_before, n_before),
    (axes[1], sss_after,  "AFTER",  t_after,  n_after),
]:
    img = ax.imshow(data, origin="lower", aspect="auto",
                    extent=[-180, 180, -90, 90],
                    cmap="RdYlBu_r", vmin=30, vmax=38, interpolation="nearest")
    plt.colorbar(img, ax=ax, label="SSS (psu)", shrink=0.7)
    ax.set_title(f"{label}  |  {t}  ({n} steps)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
plt.suptitle("SMAP SSS — Before vs After Append", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Before", "After"], [n_before, n_after], color=["steelblue", "tomato"], width=0.4)
for bar, val in zip(bars, [n_before, n_after]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(val), ha="center", va="bottom", fontsize=12)
ax.set_ylabel("Time steps")
ax.set_title(f"Store growth: +{n_after - n_before} steps")
plt.tight_layout()
plt.show()

### Verify time axis is monotonic

In [ ]:
times     = ds_after["time"].values
diffs     = np.diff(times.astype("int64"))
monotonic = bool(np.all(diffs > 0))
print(f"Monotonically increasing: {monotonic}")
print(f"Min gap: {diffs.min() / 1e9 / 86400:.1f} days")
print(f"Max gap: {diffs.max() / 1e9 / 86400:.1f} days")